# L7 · PSC 거더 전단설계 — 프리스트레스가 강도가 되는 곳

## 이 시간에 답할 질문

[L6](L6_PSC거더_휨설계.ipynb) 에서 휨을 풀었다. 거기서 프리스트레스는
**하중을 상쇄하는 역할**이었다 — 사용 시 하연 인장을 눌러 주었을 뿐,
극한 휨강도 $M_{Rd}$ 자체는 $A_p f_{pd}$ 로 정해졌다.

전단은 다르다. **프리스트레스가 콘크리트 전단강도 식에 직접 들어간다.**
이 강의에서 계산해 보면 같은 단면의 $V_{cd}$ 가 412 kN 에서
**666 kN 으로 61 % 오른다.** 프리스트레스를 무시하면 스터럽을
훨씬 많이 넣게 된다.

1. 프리스트레스는 왜 전단강도를 직접 올리는가?
2. **변각 트러스**에서 $\cot\theta$ 를 고르는 것은 설계자의 자유다.
   그 자유의 대가는 무엇인가?
3. 휨은 지간 중앙이 지배하는데 전단은 어디가 지배하는가?
4. 스터럽이 필요 없는 구간은 정말 안 넣어도 되는가?

## 근거 조문

| 내용 | 조문 |
|---|---|
| 전단철근 없는 부재의 $V_{cd}$ 식 $(4.1\text{-}7)$ | KDS 24 14 21 4.1.2.2 |
| $V_{cd}$ 하한 식 $(4.1\text{-}8)$ | KDS 24 14 21 4.1.2.2 |
| 비균열 구간 식 $(4.1\text{-}9)$ | KDS 24 14 21 4.1.2.2 |
| 변각 트러스 — 스터럽 식 $(4.1\text{-}16)$ | KDS 24 14 21 4.1.2.3 |
| 스트럿 파괴 상한 식 $(4.1\text{-}17)$ | KDS 24 14 21 4.1.2.3 |
| 압축강도 유효계수 $\nu$, $\alpha_{cw}$ 식 $(4.1\text{-}23)$ | KDS 24 14 21 4.1.2.3 |
| $1 \le \cot\theta \le 2.5$ | KDS 24 14 21 4.1.2.3 |
| 최소 전단철근과 최대 간격 | KDS 24 14 21 4.6.3 |

:::{tip}
같은 내용을 슬라이더로 움직여 보려면
[대화형 탐색기](../_static/explorer.html)를 연다. 값을 바꾸면 그래프가 바로
따라 바뀐다.
:::

## 0. 준비

**아래 코드가 하는 일** — 한글 글꼴을 등록하고 그림 색을 정한다.

In [ ]:
%matplotlib inline

import glob
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager


def use_korean_font():
    """설치된 한글 글꼴을 찾아 matplotlib 에 등록한다."""
    site = Path(sys.prefix, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
    for pattern in (
        str(site / "koreanize_matplotlib/fonts/*.ttf"),
        "/usr/share/fonts/**/*Nanum*.ttf",
        "/usr/share/fonts/**/*NotoSansCJK*.ot[fc]",
        "/usr/share/fonts/**/*NotoSansKR*.otf",
    ):
        for path in glob.glob(pattern, recursive=True):
            font_manager.fontManager.addfont(path)

    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("NanumGothic", "Malgun Gothic", "AppleGothic",
                 "Noto Sans CJK KR", "Noto Sans KR", "WenQuanYi Zen Hei"):
        if name in installed:
            plt.rcParams["font.family"] = name
            return name

    warnings.warn(
        "한글 글꼴을 찾지 못했다. 그림의 한글이 깨진다면 "
        "`pip install koreanize-matplotlib` 로 글꼴만 내려받거나, "
        "나눔고딕·Noto Sans KR 을 시스템에 설치한다.",
        stacklevel=2,
    )
    return None


print("사용 글꼴:", use_korean_font())

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 단면 분류에 쓰는 색 (압축지배 · 변화구간 · 인장지배)
C_COMP, C_TRAN, C_TENS = "#ad3327", "#b5811f", "#2a7355"
BAND = {"압축지배단면": C_COMP, "변화구간단면": C_TRAN, "인장지배단면": C_TENS}

**아래 코드가 하는 일** — 이 편에서 따라갈 설계 흐름을 순서도로
그린다. 각 단계 옆에 근거 조문을 적었다.

In [ ]:
def _use_korean_font():
    """설치된 한글 글꼴을 찾아 matplotlib 에 등록한다.

    예제 노트북의 준비 셀은 축 라벨을 ASCII 로 두므로 글꼴을 등록하지 않는다.
    순서도는 한글을 쓰므로 여기서 직접 챙긴다.
    """
    import glob
    import sys
    from pathlib import Path

    from matplotlib import font_manager

    site = Path(sys.prefix, "lib",
                f"python{sys.version_info.major}.{sys.version_info.minor}",
                "site-packages")
    for pattern in (
        str(site / "koreanize_matplotlib/fonts/*.ttf"),
        "/usr/share/fonts/**/*Nanum*.ttf",
        "/usr/share/fonts/**/*NotoSansCJK*.ot[fc]",
        "/usr/share/fonts/**/*NotoSansKR*.otf",
    ):
        for path in glob.glob(pattern, recursive=True):
            font_manager.fontManager.addfont(path)

    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("NanumGothic", "Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic"):
        if name in installed:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


def design_flowchart(title, steps, width=9.6, box_h=0.78, gap=0.30):
    """설계 흐름을 세로 순서도로 그린다.

    Args:
        title: 그림 제목
        steps: (단계 이름, KDS 조문) 튜플의 목록. 조문에 여러 개를 적으려면
            줄바꿈 대신 쉼표로 잇는다.
        width: 그림 폭 (in)
        box_h: 상자 하나의 높이 (in 환산 전 좌표 단위)
        gap: 상자 사이 간격
    """
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

    _use_korean_font()

    n = len(steps)
    fig_h = n * (box_h + gap) + 0.7
    fig, ax = plt.subplots(figsize=(width, fig_h))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, n * (box_h + gap) + 0.4)
    ax.axis("off")

    face, edge = "#eef2f8", "#1f6feb"
    for i, (name, clause) in enumerate(steps):
        y = (n - 1 - i) * (box_h + gap) + 0.2
        ax.add_patch(FancyBboxPatch(
            (0.15, y), 5.9, box_h, boxstyle="round,pad=0.04,rounding_size=0.10",
            facecolor=face, edgecolor=edge, linewidth=1.3))
        ax.text(0.42, y + box_h / 2, f"{i + 1}", va="center", ha="center",
                fontsize=10.5, color=edge, fontweight="bold")
        ax.text(0.78, y + box_h / 2, name, va="center", ha="left", fontsize=11)
        ax.text(6.25, y + box_h / 2, clause, va="center", ha="left",
                fontsize=9.5, color="#5d6675")

        if i < n - 1:
            ax.add_patch(FancyArrowPatch(
                (3.1, y), (3.1, y - gap),
                arrowstyle="-|>", mutation_scale=13,
                color=edge, linewidth=1.2))

    ax.text(0.15, n * (box_h + gap) + 0.22, title, fontsize=12.5,
            fontweight="bold", va="bottom")
    ax.text(6.25, n * (box_h + gap) + 0.24, "근거 조문", fontsize=10,
            color="#5d6675", va="bottom")
    fig.tight_layout()
    return fig

design_flowchart(
"PSC 거더 전단설계 흐름",
[
    ("설계전단력 V_Ed (극한Ⅰ)", "24 12 21 4.3 · 24 12 11 표 4.1-1"),
    ("복부 축압축 f_n = P_e/A", "24 14 21 4.1.2.2"),
    ("콘크리트 전단강도 V_cd (식 4.1-7)", "24 14 21 4.1.2.2"),
    ("전단철근 필요 구간 판정", "24 14 21 4.1.2.2"),
    ("cot θ 선택 (1 ≤ cot θ ≤ 2.5)", "24 14 21 4.1.2.3"),
    ("스트럿 상한 V_d,max (식 4.1-17)", "24 14 21 4.1.2.3"),
    ("스터럽 간격 (식 4.1-16)", "24 14 21 4.1.2.3"),
    ("최소 전단철근·최대 간격", "24 14 21 4.6.3"),
],
)
plt.show()

**아래 코드가 하는 일** — L6 에서 설계한 거더를 그대로 불러오고,
전단 검토에 필요한 값(복부 두께, 유효깊이, 복부 축압축)을 준비한다.

In [ ]:
import math

from concreteproperties_kds.kds24 import (
    COT_THETA_MAX,
    COT_THETA_MIN,
    EXAMPLE_SECTIONS,
    GAMMA_CONCRETE,
    axial_stress,
    design_concrete_shear_strength,
    design_girder,
    girder_live_load,
    max_shear_strength,
    maximum_stirrup_spacing,
    minimum_shear_reinforcement_ratio,
    required_stirrup_spacing,
    shear_reinforcement_strength,
)

# L6 에서 설계한 그 거더다
SECTION = EXAMPLE_SECTIONS["PSC-I 2.0m"]
SPAN = 30.0
STRAND, N_STRAND = 138.7, 25
FCK = 40.0
B_W = 290.0            # 복부 두께 (mm)          ← 바꿔 보라
STIRRUP_AREA = 2 * 126.7   # D13 2가닥 (mm²)    ← 바꿔 보라
F_VY = 400.0           # 스터럽 항복강도 (MPa)
COT_THETA = 2.0        # 스트럿 경사             ← 바꿔 보라

girder = design_girder(section=SECTION, span=SPAN, a_p=N_STRAND * STRAND)
props = SECTION.properties()
comp = girder.composite
D_P = girder.d_p
A_P = N_STRAND * STRAND

# 복부의 평균 축압축 — 유효 프리스트레스를 합성 단면적으로 나눈다
F_N = axial_stress(n_u=girder.p_e, a_c=comp.area, fck=FCK)

# 단위길이 하중 (kN/m)
W_TOTAL = (GAMMA_CONCRETE * props.area / 1e6
           + GAMMA_CONCRETE * 2.5 * 0.24 + 3.0)

C_CONC = "#1f7a4d"
C_STEEL = "#1f6feb"
C_LOAD = "#b3372c"
C_MUTED = "#5b6472"


def v_ed(x):
    """지점에서 x (m) 떨어진 곳의 설계전단력 (kN). 극한Ⅰ."""
    v_dc = W_TOTAL * (SPAN / 2 - x)
    v_ll = girder_live_load(span=SPAN, section=x).shear * 0.6
    return 1.25 * v_dc + 1.80 * v_ll


print(f"{SECTION.name}  지간 {SPAN:.0f} m,  강연선 {N_STRAND} 가닥")
print(f"복부 b_w = {B_W:.0f} mm,  d_p = {D_P:.0f} mm")
print(f"유효 프리스트레스 P_e = {girder.p_e / 1e3:.0f} kN")
print(f"복부 평균 축압축 f_n = {F_N:.2f} MPa")

## 1. 프리스트레스가 전단강도에 들어가는 자리

전단철근이 없는 부재의 설계전단강도는 식 $(4.1\text{-}7)$ 이다.

$$
V_{cd} = \left[ 0.85 \phi_c \kappa (\rho f_{ck})^{1/3}
+ 0.15 f_n \right] b_w d
$$

마지막 항 $0.15 f_n$ 이 핵심이다. $f_n$ 은 **단면에 걸린 평균
축압축응력**이고, 프리스트레스가 바로 그것을 만든다.

왜 축압축이 전단강도를 올리는가? 전단균열은 복부의 **주인장응력**이
콘크리트 인장강도를 넘을 때 생긴다. 축압축이 걸려 있으면 모어원이
통째로 압축 쪽으로 밀려 주인장응력이 줄어든다. 같은 전단력에서
균열이 늦게 생기는 것이다.

**아래 코드가 하는 일** — 프리스트레스를 반영할 때와 무시할 때의
$V_{cd}$ 를 비교한다.

In [ ]:
v_cd_0 = design_concrete_shear_strength(
    fck=FCK, b_w=B_W, d=D_P, a_s=A_P, f_n=0.0) / 1e3
v_cd = design_concrete_shear_strength(
    fck=FCK, b_w=B_W, d=D_P, a_s=A_P, f_n=F_N) / 1e3

print(f"프리스트레스 무시 (f_n = 0)      V_cd = {v_cd_0:7.1f} kN")
print(f"프리스트레스 반영 (f_n = {F_N:.2f})   V_cd = {v_cd:7.1f} kN")
print(f"  -> {(v_cd / v_cd_0 - 1) * 100:+.0f} %")
print()
print("f_n 에 따른 V_cd")
print(f"{'f_n (MPa)':>10} {'V_cd (kN)':>11} {'증가율':>8}")
for fn in (0.0, 1.0, 2.0, F_N, 4.0, 6.0):
    v = design_concrete_shear_strength(
        fck=FCK, b_w=B_W, d=D_P, a_s=A_P, f_n=fn) / 1e3
    mark = "  <- 이 거더" if abs(fn - F_N) < 1e-9 else ""
    print(f"{fn:10.2f} {v:11.1f} {(v / v_cd_0 - 1) * 100:7.0f} %{mark}")

**읽는 법** — $f_n$ 이 선형으로 들어가므로 증가도 선형이다.
이 거더는 $f_n = 2.79$ MPa 로 $V_{cd}$ 가 **61 % 오른다.**

여기서 주의할 것이 있다. $f_n$ 은 **유효 프리스트레스** $P_e$ 로
계산해야 한다. L6 에서 본 대로 긴장한 힘의 21 % 는 사라지므로,
긴장력 $P_{jack}$ 을 쓰면 전단강도를 과대평가한다.

## 2. 변각 트러스 — $\cot\theta$ 라는 자유

전단철근이 들어가면 KDS 24 는 **변각 트러스 모델**을 쓴다. 복부가
경사 압축 스트럿과 수직 스터럽으로 이루어진 트러스처럼 거동한다고
보는데, **스트럿의 경사각 $\theta$ 를 설계자가 고른다.**

$$
V_{sd} = \phi_s f_{vy} A_v \frac{z \cot\theta}{s}
\qquad
V_{d,\max} = \frac{\alpha_{cw} \nu \phi_c f_{ck} b_w z}
{\cot\theta + \tan\theta}
$$

$\cot\theta$ 를 키우면(스트럿을 눕히면) 한 균열을 가로지르는 스터럽이
많아져 $V_{sd}$ 가 **비례해서 커진다.** 공짜처럼 보인다.

그런데 같은 $\cot\theta$ 가 $V_{d,\max}$ 의 분모에 들어간다.
스트럿이 누울수록 **압축력이 커져 스트럿이 먼저 부서진다.**

**아래 코드가 하는 일** — 두 곡선을 함께 계산해 교차점을 찾는다.

In [ ]:
print(f"D13 2가닥 @150 mm 기준")
print(f"{'cot θ':>7} {'θ':>7} {'V_sd':>9} {'V_d,max':>10} {'지배':>12}")
print("-" * 50)
rows = []
for i in range(11):
    cot = COT_THETA_MIN + (COT_THETA_MAX - COT_THETA_MIN) * i / 10
    v_sd = shear_reinforcement_strength(
        f_vy=F_VY, a_v=STIRRUP_AREA, d=D_P, s=150.0,
        cot_theta=cot) / 1e3
    v_max = max_shear_strength(
        fck=FCK, b_w=B_W, d=D_P, cot_theta=cot) / 1e3
    gov = "스트럿" if v_sd > v_max else "스터럽"
    rows.append((cot, v_sd, v_max))
    print(f"{cot:7.2f} {math.degrees(math.atan(1 / cot)):6.1f}° "
          f"{v_sd:9.0f} {v_max:10.0f} {gov:>12}")

# 교차점
cross = None
for a, b in zip(rows, rows[1:]):
    if (a[1] - a[2]) * (b[1] - b[2]) < 0:
        t = (a[2] - a[1]) / ((b[1] - a[1]) - (b[2] - a[2]))
        cross = a[0] + t * (b[0] - a[0])
        break
print()
if cross:
    print(f"cot θ ≈ {cross:.2f} 를 넘으면 스터럽보다 스트럿이 먼저 깨진다.")
    print("그 위로는 cot θ 를 키워도 강도가 늘지 않고 오히려 준다.")

**읽는 법** — $\cot\theta$ 를 키우는 것은 공짜가 아니다. 어느 지점을
넘으면 **스트럿이 먼저 부서져** 강도가 오히려 줄어든다.

기준이 $1 \le \cot\theta \le 2.5$ 로 범위를 묶어 둔 것도 이 때문이다.
하한 1.0($45°$)은 균열 방향에서 너무 벗어나지 않게 하는 것이고,
상한 2.5($21.8°$)는 스트럿이 지나치게 눕는 것을 막는다.

> **설계의 의도** — 변각 트러스가 주는 자유는 "스터럽을 아낄 자유"가
> 아니라 **"스터럽과 복부 두께를 맞바꿀 자유"**다. $\cot\theta$ 를
> 키워 스터럽을 줄이면 복부에 더 큰 압축이 걸리므로, 복부가 얇으면
> 그 자유를 쓸 수 없다.

**아래 코드가 하는 일** — 복부 두께를 바꿔 가며 쓸 수 있는
$\cot\theta$ 의 한계를 본다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

cots = np.linspace(1.0, 2.5, 60)
v_sd_c = [shear_reinforcement_strength(f_vy=F_VY, a_v=STIRRUP_AREA,
                                       d=D_P, s=150.0, cot_theta=c) / 1e3
          for c in cots]
v_max_c = [max_shear_strength(fck=FCK, b_w=B_W, d=D_P,
                              cot_theta=c) / 1e3 for c in cots]

ax1.plot(cots, v_sd_c, color=C_STEEL, lw=2.2, label="V_sd (스터럽)")
ax1.plot(cots, v_max_c, color=C_LOAD, lw=2.2, label="V_d,max (스트럿)")
ax1.fill_between(cots, 0, np.minimum(v_sd_c, v_max_c),
                 color=C_CAP if False else "#1f7a4d", alpha=0.10,
                 label="실제 저항")
if cross:
    ax1.axvline(cross, color=C_MUTED, ls=":", lw=1.4)
    ax1.annotate(f"교차 {cross:.2f}", (cross, max(v_max_c) * 0.95),
                 ha="center", fontsize=9, color=C_MUTED)
ax1.set_xlabel("cot θ")
ax1.set_ylabel("전단강도 (kN)")
ax1.set_title("cot θ 를 키우면 스트럿이 먼저 걸린다")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# 복부 두께에 따른 V_d,max
for bw, colour in [(240.0, "#c0392b"), (290.0, "#1f6feb"),
                   (400.0, "#1f7a4d")]:
    v = [max_shear_strength(fck=FCK, b_w=bw, d=D_P, cot_theta=c) / 1e3
         for c in cots]
    ax2.plot(cots, v, lw=2.0, color=colour, label=f"b_w = {bw:.0f} mm")
ax2.plot(cots, v_sd_c, color=C_MUTED, lw=2.0, ls="--",
         label="V_sd (D13@150)")
ax2.set_xlabel("cot θ")
ax2.set_ylabel("V_d,max (kN)")
ax2.set_title("복부가 얇으면 cot θ 를 쓸 수 없다")
ax2.legend(fontsize=8.5)
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 3. 어디가 지배하는가 — 휨과 정반대다

단순 지지 보에서 휨모멘트는 **중앙**에서 최대이고 전단력은
**지점**에서 최대다. 그래서 같은 거더인데 **설계를 지배하는 위치가
정반대**다.

**아래 코드가 하는 일** — 지간을 따라 $V_{Ed}$ 와 $V_{cd}$ 를 그려
스터럽이 필요한 구간을 찾는다.

In [ ]:
print(f"{'위치 (m)':>9} {'V_Ed (kN)':>11} {'V_cd (kN)':>11} {'스터럽':>9}")
print("-" * 44)
for x in (0.0, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 12.0, 15.0):
    v = v_ed(x)
    print(f"{x:9.1f} {v:11.0f} {v_cd:11.1f} "
          f"{'필요' if v > v_cd else '불필요':>9}")

# 스터럽이 필요 없어지는 위치
xs = np.linspace(0, SPAN / 2, 400)
vs = np.array([v_ed(float(x)) for x in xs])
idx = np.argmax(vs <= v_cd)
x_free = float(xs[idx]) if vs[idx] <= v_cd else None
print()
if x_free:
    print(f"x ≈ {x_free:.1f} m 부터 계산상 스터럽이 필요 없다 "
          f"(지간의 {x_free / SPAN * 100:.0f} %).")
    print("다만 '필요 없다'와 '넣지 않는다'는 다르다 — 5절을 볼 것.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))

ax.plot(xs, vs, color=C_LOAD, lw=2.4, label="V_Ed (극한Ⅰ)")
ax.axhline(v_cd, color=C_CONC, lw=2.2, label=f"V_cd = {v_cd:.0f} kN")
ax.fill_between(xs, v_cd, vs, where=(vs > v_cd), color=C_LOAD,
                alpha=0.14, label="스터럽이 받아야 할 몫")
if x_free:
    ax.axvline(x_free, color=C_MUTED, ls=":", lw=1.4)
    ax.annotate(f"x = {x_free:.1f} m", (x_free, vs.max() * 0.9),
                ha="center", fontsize=9, color=C_MUTED)

# 참고 — 휨모멘트는 반대로 중앙이 최대다
ax2 = ax.twinx()
m = [1.25 * W_TOTAL * x * (SPAN - x) / 2
     + 1.80 * girder_live_load(span=SPAN).moment * 0.6
     * (4 * x * (SPAN - x) / SPAN**2) for x in xs]
ax2.plot(xs, m, color=C_MUTED, lw=1.6, ls="--", alpha=0.75,
         label="M_Ed (참고, 오른쪽 축)")
ax2.set_ylabel("휨모멘트 (kN·m)", color=C_MUTED)
ax2.tick_params(axis="y", labelcolor=C_MUTED)

ax.set_xlabel("지점에서의 거리 (m)")
ax.set_ylabel("전단력 (kN)")
ax.set_title("전단은 지점이, 휨은 중앙이 지배한다")
ax.set_xlim(0, SPAN / 2)
ax.set_ylim(0, vs.max() * 1.1)
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**읽는 법** — 두 곡선이 정확히 반대로 간다. 붉은 영역이 스터럽이
받아야 할 몫이고, 지간의 약 30 % 구간에만 나타난다.

이것이 PSC 거더의 배근이 **단부에 촘촘하고 중앙에 성긴** 이유다.
그리고 L6 에서 본 대로 **텐던은 반대로** 중앙에서 편심이 크고 단부에서
핵 안으로 들어온다. 두 배근이 서로 어긋나 있는 셈이다.

## 4. 스터럽 배치

**아래 코드가 하는 일** — 위치별로 필요한 스터럽 간격을 구하고,
최소 전단철근·최대 간격 규정과 견주어 채택값을 정한다.

In [ ]:
s_max = maximum_stirrup_spacing(d=D_P)
rho_min = minimum_shear_reinforcement_ratio(fck=FCK, f_y=F_VY)
s_rho = STIRRUP_AREA / (rho_min * B_W)

print(f"최대 간격 규정        {s_max:.0f} mm")
print(f"최소 전단철근비       {rho_min * 100:.3f} %  ->  간격 {s_rho:.0f} mm 이하")
print(f"실제 상한 = min       {min(s_max, s_rho):.0f} mm")
print()
print(f"{'위치':>6} {'V_Ed':>8} {'V_sd 소요':>10} {'필요 간격':>10} "
      f"{'채택':>8}")
print("-" * 48)
layout = []
for x in (0, 1, 2, 3, 4, 5, 7.5, 10, 12, 15):
    v = v_ed(float(x))
    need = v - v_cd
    if need <= 0:
        s_adopt = min(s_max, s_rho)
        print(f"{x:6.1f} {v:8.0f} {'-':>10} {'불필요':>10} "
              f"{s_adopt:8.0f}")
    else:
        s_req = required_stirrup_spacing(
            v_ed=need * 1e3, d=D_P, a_v=STIRRUP_AREA,
            cot_theta=COT_THETA)
        s_adopt = min(s_req, s_max, s_rho)
        print(f"{x:6.1f} {v:8.0f} {need:10.0f} {s_req:10.0f} "
              f"{s_adopt:8.0f}")
    layout.append((x, s_adopt))

**읽는 법** — 지점에서 약 510 mm, 중앙부에서는 **최소 전단철근이
정하는 691 mm** 다. 계산상 필요 없는 구간에서도 간격의 상한이 걸리는
것이다.

## 5. "필요 없다"와 "넣지 않는다"는 다르다

3절에서 지간의 70 % 는 계산상 스터럽이 필요 없다고 나왔다. 그렇다고
정말 안 넣지는 않는다. 기준이 **최소 전단철근**을 요구하기 때문이다.

이유는 전단 파괴의 성격에 있다. 휨 파괴는 철근이 항복하며 처짐이
크게 자라 **예고**가 있지만, 전단 파괴는 사인장균열이 갑자기 열리며
**예고 없이** 온다. $V_{cd}$ 식 자체가 실험의 회귀식이라 흩어짐도 크다.

> **설계의 의도** — 최소 전단철근은 강도를 위한 것이 아니라 **취성
> 파괴를 막기 위한 것**이다. $V_{Ed}$ 가 $V_{cd}$ 보다 작아도, 계산이
> 빗나갔을 때 부재가 조용히 무너지지 않도록 붙잡아 둔다.

## 6. KDS 14 로 풀면

같은 단면을 강도설계법으로 풀면 얼마나 다른가?

**아래 코드가 하는 일** — KDS 14 20 22 의 $V_c$ 와 견준다. KDS 14 는
프리스트레스를 $V_c$ 식에 다른 방식으로 반영하므로, 여기서는
**전단철근이 없을 때의 콘크리트 몫**만 형식적으로 견준다.

In [ ]:
# KDS 14 — 간이식 V_c = (1/6) sqrt(f_ck) b_w d, phi = 0.75
v_c_14 = 0.75 * (1 / 6) * math.sqrt(FCK) * B_W * D_P / 1e3

print(f"KDS 14  φV_c = 0.75 x (1/6)√f_ck b_w d = {v_c_14:7.1f} kN")
print(f"KDS 24  V_cd (f_n = 0)                 = {v_cd_0:7.1f} kN")
print(f"KDS 24  V_cd (f_n = {F_N:.2f})              = {v_cd:7.1f} kN")
print()
print(f"프리스트레스를 빼면 KDS 24 가 KDS 14 의 "
      f"{v_cd_0 / v_c_14 * 100:.0f} % 로 낮고,")
print(f"넣으면 {v_cd / v_c_14 * 100:.0f} % 로 뒤집힌다.")
print()
print("KDS 14 의 간이식은 f_ck 만 보고 철근비도 축응력도 보지 않는다.")
print("KDS 24 는 ρ 와 f_n 을 모두 넣어 부재의 조건을 반영한다.")

**읽는 법** — 프리스트레스를 무시하면 KDS 24 가 더 보수적인데,
반영하면 뒤집힌다. **PSC 부재에서 두 기준의 차이는 축응력 항이
만든다.**

(KDS 14 20 60 은 PSC 부재에 대해 $V_{ci}$ / $V_{cw}$ 를 따로 두는
상세식을 갖고 있다. 위 비교는 간이식만 형식적으로 견준 것이므로
실제 KDS 14 설계값과는 다르다.)

## 7. 바꿔 보며 확인할 것

1. `B_W` 를 240 mm(EX거더 최소 복부두께)로 줄이면 쓸 수 있는
   $\cot\theta$ 의 상한이 얼마나 내려가는가?
2. `COT_THETA` 를 2.5 로 올리면 스터럽 간격이 얼마나 벌어지는가?
   그때 $V_{d,\max}$ 가 $V_{Ed}$ 를 여전히 넘는가?
3. `STIRRUP_AREA` 를 D16 2가닥으로 올리면 지점부 간격이 얼마가 되는가?
4. `N_STRAND` 를 줄여 $f_n$ 을 낮추면 스터럽이 필요한 구간이
   얼마나 길어지는가?

## 8. 정리

1. **프리스트레스는 전단강도에 직접 들어간다.** 식 $(4.1\text{-}7)$ 의
   $0.15 f_n$ 항이 이 거더에서 $V_{cd}$ 를 **61 % 올린다.** 휨에서는
   없던 효과다.
2. **$f_n$ 은 유효 프리스트레스로 계산해야 한다.** 손실 21 % 를
   빠뜨리면 전단강도를 과대평가한다.
3. **$\cot\theta$ 는 공짜가 아니다.** 키우면 스터럽이 줄지만 스트럿
   압축이 커져, 어느 지점을 넘으면 스트럿이 먼저 부서진다.
   복부가 얇을수록 그 지점이 빨리 온다.
4. **전단은 지점이, 휨은 중앙이 지배한다.** 스터럽은 단부에 촘촘하고
   텐던 편심은 중앙에서 크다 — 두 배근이 서로 어긋난다.
5. **계산상 필요 없어도 최소 전단철근은 넣는다.** 전단 파괴는 예고가
   없기 때문이다.

## 9. 생각해 볼 문제

1. $V_{cd}$ 식의 $f_n$ 항은 축압축이 클수록 무한정 커지는가? 기준이
   상한을 두는 이유는 무엇일까?
2. 텐던을 드레이프하면 단부에서 텐던이 위로 올라간다. 이때 텐던의
   **수직 성분**이 전단력을 직접 덜어 준다. 이 강의는 그 효과를
   넣지 않았는데, 넣으면 결과가 어떻게 달라지겠는가?
3. 최소 전단철근이 취성 파괴를 막기 위한 것이라면, 그 양은 무엇을
   기준으로 정해야 하는가? 현재 규정은 $f_{ck}$ 와 $f_y$ 만 본다.
4. 합성 거더에서 바닥판과 거더 사이의 **수평전단**은 이 강의가 다루지
   않았다. 그 검토가 왜 따로 필요한가?
5. 지점 근처에서는 하중이 스트럿을 통해 지점으로 직접 흐른다
   (아치작용). 기준이 지점에서 $d$ 이내를 달리 취급하는 근거는
   무엇인가?